In [1]:
# ============================================================================
# Windows 多进程问题修复
# ============================================================================
import multiprocessing
import sys
import warnings
warnings.filterwarnings('ignore')

# Windows 多进程配置
if sys.platform == 'win32':
    try:
        multiprocessing.set_start_method('spawn')
        print('多进程启动方法设置为 spawn (Windows兼容性)')
    except RuntimeError:
        pass  # 已经设置过

多进程启动方法设置为 spawn (Windows兼容性)


In [2]:
# 1. 导入必要的库
# vnpy相关
from vnpy.trader.setting import SETTINGS
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import  HistoryRequest
from vnpy.trader.datafeed import get_datafeed
from vnpy.alpha.lab import AlphaLab
from vnpy.alpha.dataset import AlphaDataset
from vnpy.alpha.dataset.processor import (
    process_drop_na, process_fill_na, process_cs_norm
)

# 数据处理
import polars as pl

print('导入完成')

导入完成


In [4]:
# 2. 配置RQData数据服务

# 检查是否已配置用户名密码
print('当前数据服务配置:')
print('datafeed.username:', SETTINGS.get('datafeed.username', '未设置'))
print('datafeed.password:', SETTINGS.get('datafeed.password', '未设置'))

# 初始化数据服务
datafeed = get_datafeed()
print(f'数据服务类型: {datafeed.__class__.__name__}')

# 尝试初始化
inited = datafeed.init(output=print)
print(f'初始化结果: {inited}')


当前数据服务配置:
datafeed.username: license
datafeed.password: GXgIL2zU_QvKOsm3Lq6Rsf2mVGHlt6YWz578U9f2SpSgQ01VQAqqoxgCMKgc7KZLRUI0lHoMQGszcYVmBDlNhcRuPaluDhh3f1_cJZ20qaSuTia581eHR_-EfSskCimFRRdcydEP7a1m-0WZY6vW9WAPnslg3rrE0mbySAckyCk=R3M8eOvPwDDgMV9d9saPYFxR1ws82HWZWoHJENwz4wJsOU1G4tIlcuBWB5684ArTMIJTef6lgSYeiq9l9CHOR_nfvsyVHG3hsjlkiG8aiBaP8jA9EMJGFF1Gd3yYR-6fTy1mi1Wr1PU_g1HRDQHe1h8OflFJ2FpgT49swSw66FM=
数据服务类型: RqdataDatafeed
初始化结果: True


In [5]:
# 3. 创建AlphaLab实验室
import os
lab_path = os.path.join(os.getcwd(), 'lab2')
lab = AlphaLab(lab_path)
print(f'AlphaLab路径: {lab_path}')

AlphaLab路径: D:\Aquant project\MF\lab2


In [9]:
# 5. 下载数据
index_symbol = "000300.SSE"
rq_index_symbol = "000300.XSHG"
extended_days: int = 0
# 时间范围：2023-01-01 至 2026-03-27
from datetime import datetime
test_start = datetime(2025, 1, 1)
test_end = datetime(2026, 3, 27)
start = datetime(2013, 1, 1)
end = datetime(2026, 3, 27)
interval = Interval.DAILY                  #数据频率

import rqdatac as rq
# 下载指数成分股 时间对应的成分股列表
data = rq.index_components(rq_index_symbol, start_date=start, end_date=end)
# 转换合约代码
index_components = {}
for dt, rq_symbols in data.items():
    vt_symbols: list = []

    for rq_symbol in rq_symbols:
        vt_symbol = rq_symbol.replace("XSHG", "SSE").replace("XSHE", "SZSE")
        vt_symbols.append(vt_symbol)

    index_components[dt.strftime("%Y-%m-%d")] = vt_symbols    #index_components = {"%Y-%m-%d":[成分股列表]，....} vnpy代码格式


# 保存到数据中心
lab.save_component_data(index_symbol, index_components)

component_symbols = lab.load_component_symbols(index_symbol, start, end)
print(component_symbols)


['002422.SZSE', '600682.SSE', '000783.SZSE', '600837.SSE', '601225.SSE', '601666.SSE', '000630.SZSE', '000536.SZSE', '002601.SZSE', '600143.SSE', '002797.SZSE', '600085.SSE', '001289.SZSE', '600783.SSE', '000596.SZSE', '002153.SZSE', '300072.SZSE', '000528.SZSE', '600905.SSE', '601698.SSE', '600930.SSE', '002600.SZSE', '600436.SSE', '600060.SSE', '002450.SZSE', '605499.SSE', '688256.SSE', '603885.SSE', '600221.SSE', '300760.SZSE', '600098.SSE', '300888.SZSE', '600660.SSE', '601808.SSE', '600369.SSE', '300498.SZSE', '603160.SSE', '002007.SZSE', '603659.SSE', '002958.SZSE', '002378.SZSE', '000969.SZSE', '601288.SSE', '601717.SSE', '601929.SSE', '600871.SSE', '002294.SZSE', '300769.SZSE', '600522.SSE', '002572.SZSE', '300308.SZSE', '601318.SSE', '000413.SZSE', '600816.SSE', '002352.SZSE', '601689.SSE', '601169.SSE', '600340.SSE', '000539.SZSE', '600566.SSE', '600331.SSE', '601333.SSE', '601328.SSE', '601369.SSE', '002468.SZSE', '601577.SSE', '300015.SZSE', '001965.SZSE', '000538.SZSE', '0

In [10]:
print(len(component_symbols))

714


In [11]:
# 转换时间格式
from vnpy.trader.database import DB_TZ
from vnpy.alpha import  logger
from tqdm import tqdm

start = start.replace(tzinfo=DB_TZ) #下载的时候要有时区  下下来后没有时区了
end = end.replace(tzinfo=DB_TZ)

# 除了成分股，还要下载指数数据
task_symbols = component_symbols

# 遍历下载数据


from tqdm import tqdm

n = 0
for vt_symbol in tqdm(task_symbols):
    symbol, exchange_str = vt_symbol.split(".")
    print(f'下载 {vt_symbol} ...')
    req = HistoryRequest(symbol, Exchange(exchange_str), start, end, interval)
    bars = datafeed.query_bar_history(req)

    if bars:
        lab.save_bar_data(bars)
        n+=1
        print(f'  -> 保存 {len(bars)} 条K线')
    else:
        logger.error(f"下载{vt_symbol}数据失败")

  0%|          | 0/714 [00:00<?, ?it/s]

下载 002422.SZSE ...


  0%|          | 2/714 [00:00<03:05,  3.83it/s]

  -> 保存 3210 条K线
下载 600682.SSE ...
  -> 保存 3210 条K线
下载 000783.SZSE ...


  1%|          | 4/714 [00:00<02:14,  5.29it/s]

...

  -> 保存 2714 条K线
下载 601969.SSE ...


100%|█████████▉| 712/714 [01:55<00:00,  5.00it/s]

  -> 保存 2744 条K线
下载 601066.SSE ...
  -> 保存 1884 条K线
下载 002709.SZSE ...


100%|██████████| 714/714 [01:56<00:00,  6.14it/s]

  -> 保存 2957 条K线
下载 300661.SZSE ...
  -> 保存 2139 条K线


In [13]:
import rqdatac as rq
quota = rq.user.get_quota()
print(quota)

{'bytes_used': 61234665, 'bytes_limit': 209715200.0, 'remaining_days': 451, 'license_type': 'EDU'}


In [14]:
symbol, exchange_str = index_symbol.split(".")
print(f'下载 {index_symbol} ...')
req = HistoryRequest(symbol, Exchange(exchange_str), start, end, interval)
bars = datafeed.query_bar_history(req)

if bars:
    lab.save_bar_data(bars)
    n+=1
    print(f'  -> 保存 {len(bars)} 条K线')
else:
    logger.error(f"下载{vt_symbol}数据失败")

下载 000300.SSE ...
  -> 保存 3210 条K线
